In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Mapping 19 labels into 6

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
import numpy as np
import os
import wandb

# os.environ["WANDB_API_KEY"] = # Add API key
# os.environ["WANDB_PROJECT"] = # Add project name

ADVERSE_LABELS = [
    'adverse_employment', 'adverse_housing', 'adverse_transportation',
    'adverse_parent', 'adverse_relationship', 'adverse_support'
]

ADVERSE_MAPPING = {
    'adverse_employment': ['EMPLOYMENT_unemployed', 'EMPLOYMENT_underemployed', 'EMPLOYMENT_disability'],
    'adverse_housing': ['HOUSING_poor', 'HOUSING_undomiciled', 'HOUSING_other'],
    'adverse_transportation': ['TRANSPORTATION_distance', 'TRANSPORTATION_resource', 'TRANSPORTATION_other'],
    'adverse_parent': ['PARENT'],
    'adverse_relationship': ['RELATIONSHIP_widowed', 'RELATIONSHIP_divorced', 'RELATIONSHIP_single'],
    'adverse_support': ['SUPPORT_minus']
}

SYNTHETIC_LABEL_MAP = {
    'employment': 'adverse_employment', 'housing': 'adverse_housing',
    'transportation': 'adverse_transportation', 'parent': 'adverse_parent',
    'relationship': 'adverse_relationship', 'support': 'adverse_support'
}

Mapping 19 columns into 6

In [ ]:
def create_adverse_labels_from_mimic(df):
    df_out = pd.DataFrame(index=df.index)
    df_out['text'] = df['text']

    for adv_label, source_cols in ADVERSE_MAPPING.items():
        numeric_cols = pd.DataFrame()
        for col in source_cols:
            if col in df.columns:
                 numeric_cols[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols = numeric_cols.fillna(0)
        df_out[adv_label] = (numeric_cols.sum(axis=1) > 0).astype(float)

    df_out['labels'] = df_out[ADVERSE_LABELS].values.tolist()
    return df_out[['text', 'labels']].dropna(subset=['text'])

def load_and_format_synthetic_data(filenames):
    df_list = []
    base_path = '/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/'

    for f in filenames:
        file_path = os.path.join(base_path, f)
        try:
            df_list.append(pd.read_csv(file_path))
            print(f"Successfully loaded {file_path}")
        except FileNotFoundError:
            print(f"Warning: Synthetic file {file_path} not found.")
        except Exception:
            if 'SyntheticSentencs_Round2' in f:
                try:
                    typo_path = os.path.join(base_path, 'SyntheticSentencs_Round2.csv')
                    df_list.append(pd.read_csv(typo_path))
                    print(f"Successfully loaded {typo_path} (via typo fix)")
                except: pass

    if not df_list:
        return pd.DataFrame(columns=['text', 'labels'])

    df_synthetic = pd.concat(df_list)
    df_synthetic = df_synthetic.dropna(subset=['text', 'label', 'adverse'])

    df_out = pd.DataFrame(0.0, index=df_synthetic.index, columns=ADVERSE_LABELS)
    df_out['text'] = df_synthetic['text']

    for i, row in df_synthetic.iterrows():
        labels = str(row['label']).split(',')
        adverse_statuses = str(row['adverse']).split(',')

        for j, label_str in enumerate(labels):
            if j >= len(adverse_statuses): continue
            label_str = label_str.strip().lower()
            adverse_str = adverse_statuses[j].strip().lower()

            if adverse_str == 'adverse' and label_str in SYNTHETIC_LABEL_MAP:
                target_label = SYNTHETIC_LABEL_MAP[label_str]
                df_out.at[i, target_label] = 1.0

    df_out['labels'] = df_out[ADVERSE_LABELS].values.tolist()
    return df_out[['text', 'labels']]

Loading and splitting data

In [ ]:
try:
    df_mimic = pd.read_csv('/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SDOH_MIMICIII_physio_release.csv')
    df_mimic_adverse = create_adverse_labels_from_mimic(df_mimic)
    print(f"Loaded {len(df_mimic_adverse)} real MIMIC sentences.")
except FileNotFoundError:
    print("Error: Could not find SDOH_MIMICIII_physio_release.csv")

train_val_df, test_df = train_test_split(df_mimic_adverse, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

print(f"Train Size: {len(train_df)} | Val Size: {len(val_df)} | Test Size: {len(test_df)}")

Loaded 5321 real MIMIC sentences.
Train Size: 3192 | Val Size: 1064 | Test Size: 1065


Calculating Class Weights

In [ ]:
labels_array = np.array(train_df['labels'].tolist())
pos_counts = np.sum(labels_array, axis=0)
neg_counts = len(train_df) - pos_counts
pos_weight = (neg_counts + 1) / (pos_counts + 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float).to(device)
print(f"Weights calculated for {len(pos_weight)} labels.")

Weights calculated for 6 labels.


Adding synthetic data

In [ ]:
synthetic_files = [
    'ManuallyAnnotatedSyntheticSentences.csv',
    'SyntheticSentences_Round1.csv',
    'SyntheticSentencs_Round2.csv'
]
df_synthetic_adverse = load_and_format_synthetic_data(synthetic_files)

train_df_augmented = pd.concat([train_df, df_synthetic_adverse]).sample(frac=1)
print(f"Final Augmented Train Size: {len(train_df_augmented)}")

train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/ManuallyAnnotatedSyntheticSentences.csv
Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SyntheticSentences_Round1.csv
Successfully loaded /content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SyntheticSentencs_Round2.csv
Final Augmented Train Size: 5472


Model Initialisation with pretrained weights

In [ ]:

model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(ADVERSE_LABELS),
    problem_type="multi_label_classification"
)

for param in model.bert.embeddings.parameters():
    param.requires_grad = False
for layer in model.bert.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=512)

train_dataset_tokenized = train_dataset.map(tokenize_function, batched=True)
val_dataset_tokenized = val_dataset.map(tokenize_function, batched=True)
test_dataset_tokenized = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5472 [00:00<?, ? examples/s]

Map:   0%|          | 0/1064 [00:00<?, ? examples/s]

Map:   0%|          | 0/1065 [00:00<?, ? examples/s]

Training

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score
import torch

def compute_metrics_per_label(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))

    best_thresholds = []
    final_preds = np.zeros(probs.shape)

    for i in range(labels.shape[1]):
        best_f1 = 0
        best_thresh = 0.5
        for thresh in np.arange(0.1, 0.9, 0.05):
            current_preds = (probs[:, i] > thresh).astype(int)
            current_f1 = f1_score(labels[:, i], current_preds, zero_division=0)
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        best_thresholds.append(round(float(best_thresh), 3))
        final_preds[:, i] = (probs[:, i] > best_thresh).astype(int)

    f1_micro = f1_score(labels, final_preds, average='micro', zero_division=0)
    f1_macro = f1_score(labels, final_preds, average='macro', zero_division=0)
    roc_auc_micro = roc_auc_score(labels, probs, average='micro')

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'roc_auc_micro': roc_auc_micro,
        'best_thresholds': str(best_thresholds)
    }

class ASLTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')

        probs = torch.sigmoid(logits)

        loss_pos = labels * torch.log(probs + 1e-8)
        loss_neg = (1 - labels) * torch.log(1 - probs + 1e-8) * (probs ** 2)

        loss = -(loss_pos + loss_neg).mean()

        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./results_adverse_sdoh_aug',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs_adverse_sdoh_aug',
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=1,
    report_to="wandb"
)

trainer = ASLTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    compute_metrics=compute_metrics_per_label,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

test_results = trainer.evaluate(test_dataset_tokenized)
print(f"\n--- Final Test Results ---")
print(f"Micro F1-Score: {test_results['eval_f1_micro']:.4f}")
print(f"Macro F1-Score: {test_results['eval_f1_macro']:.4f}")
print(f"Micro ROC-AUC:  {test_results['eval_roc_auc_micro']:.4f}")
print(f"Thresholds:     {test_results['eval_best_thresholds']}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Roc Auc Micro,Best Thresholds
1,0.173937,0.007364,0.072727,0.140946,0.957233,"[0.1, 0.15, 0.5, 0.35, 0.5, 0.4]"
2,0.152821,0.007126,0.116959,0.140885,0.956117,"[0.15, 0.1, 0.5, 0.25, 0.3, 0.35]"
3,0.140044,0.008397,0.080808,0.146073,0.957217,"[0.1, 0.1, 0.5, 0.65, 0.4, 0.55]"
4,0.131602,0.007613,0.196429,0.153780,0.946947,"[0.5, 0.1, 0.5, 0.15, 0.25, 0.45]"
5,0.141938,0.007493,0.191781,0.163139,0.957556,"[0.5, 0.1, 0.5, 0.3, 0.5, 0.6]"
6,0.137964,0.007599,0.181818,0.171522,0.948906,"[0.5, 0.1, 0.5, 0.4, 0.55, 0.6]"
7,0.131580,0.008579,0.260870,0.163743,0.945748,"[0.5, 0.5, 0.5, 0.4, 0.65, 0.75]"
8,0.129417,0.007896,0.263158,0.169841,0.938703,"[0.5, 0.5, 0.5, 0.55, 0.6, 0.65]"
9,0.124215,0.007786,0.186667,0.176768,0.928962,"[0.5, 0.1, 0.5, 0.35, 0.5, 0.7]"
10,0.123221,0.008067,0.173913,0.178743,0.934593,"[0.5, 0.1, 0.5, 0.55, 0.55, 0.65]"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


--- Final Test Results ---
Micro F1-Score: 0.4444
Macro F1-Score: 0.3891
Micro ROC-AUC:  0.9676
Thresholds:     [0.55, 0.2, 0.5, 0.8, 0.5, 0.65]


In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

print(f"Vocab Size: {config.vocab_size}")
print(f"Hidden Size: {config.hidden_size}")
print(f"Layers: {config.num_hidden_layers}")
print(f"Max Position Embeddings: {config.max_position_embeddings}")

Vocab Size: 28996
Hidden Size: 768
Layers: 12
Max Position Embeddings: 512


In [ ]:
from transformers import AutoModel

model_name = "emilyalsentzer/Bio_ClinicalBERT"
model = AutoModel.from_pretrained(model_name)

for param in model.embeddings.parameters():
    param.requires_grad = False
for layer in model.encoder.layer[:10]:
    for param in layer.parameters():
        param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"--- Model Statistics for {model_name} ---")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.1f}%)")
print(f"Frozen Parameters:    {frozen_params:,}")

model_size_mb = total_params * 4 / (1024 * 1024)
print(f"Approximate Model Size: {model_size_mb:.2f} MB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Model Statistics for emilyalsentzer/Bio_ClinicalBERT ---
Total Parameters:     108,310,272
Trainable Parameters: 14,766,336 (13.6%)
Frozen Parameters:    93,543,936
Approximate Model Size: 413.17 MB


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv('/content/drive/MyDrive/annotation-dataset-of-social-determinants-of-health-from-mimic-iii-clinical-care-database-1.0.1/SDOH_MIMICIII_physio_release.csv')
except FileNotFoundError:
    print("Error: File not found. Using local file name for demonstration.")
    df = pd.read_csv('SDOH_MIMICIII_physio_release.csv')
df_clean = df.dropna(subset=['text'])

train_val_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

total_patients = df_clean['patient_id'].nunique()
train_patients_count = train_df['patient_id'].nunique()
val_patients_count = val_df['patient_id'].nunique()
test_patients_count = test_df['patient_id'].nunique()

print("--- Patient Population Statistics ---")
print(f"Total Unique Patients in Dataset: {total_patients}")
print(f"Unique Patients in Training Set:  {train_patients_count}")
print(f"Unique Patients in Validation Set:{val_patients_count}")
print(f"Unique Patients in Testing Set:   {test_patients_count}")

--- Patient Population Statistics ---
Total Unique Patients in Dataset: 183
Unique Patients in Training Set:  183
Unique Patients in Validation Set:177
Unique Patients in Testing Set:   177
